# 02 · Cleaning + simulated OBD-II features

Applies the cleaning contract and synthesises `battery_soh` and `trans_adapt_offset` (see spec). Logic lives in `ml.ingest` / `ml.features` — this notebook narrates and visualises it.

In [ ]:
import sys; sys.path.insert(0, '..' if __import__('pathlib').Path('..').joinpath('app').exists() else '../..')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest
sns.set_theme()
raw = ingest.load_raw(); df = ingest.build_engineered_csv()
print('raw:', raw.shape, '-> engineered:', df.shape); df.head()

### Isolating cleaning from feature engineering
`ingest.clean()` and `ingest.clean_and_engineer()` are the two building blocks behind
`build_engineered_csv()` above -- useful when you want to experiment with just the
cleaning rules (row-drop filters, `fx_rate`) or just the feature formulas in isolation.

In [ ]:
from app.config import get_settings
fx_rate = get_settings().fx_gbp_to_rm
cleaned_only = ingest.clean(raw, fx_rate=fx_rate)
print('raw:', raw.shape, '-> cleaned (pre-engineering):', cleaned_only.shape)
print('dropped:', len(raw) - len(cleaned_only), 'rows (nulls/dupes/outliers -- see ml.ingest.clean)')
cleaned_only.head()

In [ ]:
manual_engineered = ingest.clean_and_engineer(raw, fx_rate=fx_rate, seed=42)
assert manual_engineered.equals(df), 'clean_and_engineer(seed=42) must match build_engineered_csv() exactly'
print('clean_and_engineer == clean() + add_engineered_features(), matches build_engineered_csv() output')
manual_engineered.head()

### `battery_soh` — Starter Battery State of Health
Time-dominant for petrol/diesel; non-linear in mileage for hybrids.

In [ ]:
from ml.features import battery_soh_base, SOH_FLOOR, SOH_NOISE_STD
import numpy as np
ages = np.arange(0, 21)
for ft in ['Petrol', 'Diesel', 'Hybrid']:
    vals = [battery_soh_base(ft, a, 60_000) for a in ages]
    plt.plot(ages, vals, label=ft, marker='.')
plt.axhline(SOH_FLOOR, color='gray', ls='--', label=f'SOH_FLOOR={SOH_FLOOR}')
plt.xlabel('age (years)'); plt.ylabel('battery_soh_base (%, unclipped)')
plt.title(f'deterministic base curve @ mileage=60,000 (noise_std={SOH_NOISE_STD} not applied here)')
plt.legend(); plt.show()

In [ ]:
import numpy as np
mileages = np.arange(0, 300_001, 10_000)
for ft in ['Petrol', 'Hybrid']:
    vals = [battery_soh_base(ft, 5, m) for m in mileages]
    plt.plot(mileages, vals, label=ft, marker='.')
plt.xlabel('mileage'); plt.ylabel('battery_soh_base (%, unclipped)')
plt.title('@ age=5 -- Hybrid should bend down non-linearly with mileage')
plt.legend(); plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
df['battery_soh'].plot.hist(bins=40, ax=ax[0], title='battery_soh (%)')
sns.scatterplot(data=df.sample(min(2000,len(df)), random_state=0), x='mileage', y='battery_soh',
                hue='fuel_type', s=10, ax=ax[1]); ax[1].set_title('SoH vs mileage'); plt.tight_layout()

### `trans_adapt_offset` — Transmission Adaptation Offset
Manual is exactly 0 (own branch); automatics are strictly negative and scale with mileage.

In [ ]:
from ml.features import trans_adapt_offset_base
mileages2 = np.arange(0, 300_001, 5_000)
for t in ['Manual', 'Automatic', 'Semi-Auto']:
    vals = [trans_adapt_offset_base(t, m) for m in mileages2]
    plt.plot(mileages2, vals, label=t, marker='.')
plt.axvline(60_000, color='gray', ls=':'); plt.axvline(120_000, color='gray', ls=':')
plt.xlabel('mileage'); plt.ylabel('trans_adapt_offset_base (deterministic, no noise)')
plt.title('wear bands: normal / noticeable / critical (dotted lines = band boundaries)')
plt.legend(); plt.show()

In [ ]:
print(df.groupby('transmission')['trans_adapt_offset'].describe()[['mean','min','max']])
auto = df[df['transmission'] != 'Manual'].sample(min(2000,len(df)), random_state=0)
sns.scatterplot(data=auto, x='mileage', y='trans_adapt_offset', s=10)
plt.title('offset vs mileage (non-manual)')

In [ ]:
from ml.features import engineer_profile
demo_profile = {'fuel_type': 'Hybrid', 'transmission': 'Automatic', 'age': 5, 'mileage': 70_000}
print('engineer_profile ->', engineer_profile(demo_profile))
print('same call again (must be identical -- noise-free) ->', engineer_profile(demo_profile))

### Experiment here
Tweak `battery_soh_base` / `trans_adapt_offset_base` in `ml/features.py` (or the noise/floor
constants above), then re-run this notebook top-to-bottom -- `ingest.build_engineered_csv()`
regenerates `data/merc_engineered.csv` -- before jumping to `03_modeling.ipynb` to see the
downstream effect on feature importance and model metrics.

Engineered dataset written to `data/merc_engineered.csv` — this is what the model trains on.